# Export your trained PharmaVision EfficientNetV2-S to ONNX

**Purpose:** Convert your own `best_efficientnetv2_s.pth` Kaggle model into the `efficientnetv2_s.onnx` file needed by the GitHub/Vercel website. **This DOES NOT retrain the network.**

1. **Add Input** → add your private `PharmaVision Trained Model Backup` dataset containing `best_efficientnetv2_s.pth`.
2. Run these cells from top to bottom. You can use CPU; GPU is not required.
3. Download the exported `.onnx` file and place it inside `public/model/` of the website repository.

Only load checkpoints you created or trust, because PyTorch checkpoint formats may execute Python during loading. The resulting ONNX file is **not** the research authors' original implementation; it contains your reconstruction's weights.


## 1. Install ONNX tools (one-time per Kaggle session)

In [ ]:
%pip install -q onnx onnxruntime

## 2. Locate the existing checkpoint (no training)

In [ ]:
from pathlib import Path

PREFERRED = Path('/kaggle/working/pharmavision_reconstruction/best_efficientnetv2_s.pth')

# Change this if your checkpoint is in a different location.
CHECKPOINT_OVERRIDE = None

if CHECKPOINT_OVERRIDE:
    checkpoint_path = Path(CHECKPOINT_OVERRIDE)
elif PREFERRED.is_file():
    checkpoint_path = PREFERRED
else:
    matches = sorted(Path('/kaggle/input').rglob('best_efficientnetv2_s.pth'))
    if not matches:
        raise FileNotFoundError(
            'Checkpoint missing. In Kaggle click Add Input and attach your private backup '
            'containing best_efficientnetv2_s.pth, then rerun this cell.'
        )
    if len(matches) > 1:
        print('More than one checkpoint found. Using the first; verify this is your epoch-18 model:')
        for p in matches:
            print(' -', p)
    checkpoint_path = matches[0]

print('Checkpoint found:', checkpoint_path)
print('Size:', round(checkpoint_path.stat().st_size / (1024 ** 2), 1), 'MiB')


## 3. Rebuild EfficientNetV2-S and export an ONNX model

In [ ]:
import torch
from torch import nn
from torchvision.models import efficientnet_v2_s
import onnx

# Trust this file only if it is the checkpoint you created in Kaggle.
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print('Checkpoint epoch:', checkpoint.get('epoch'))
print('Class indices:', checkpoint.get('classes'))
assert checkpoint.get('classes') == {'defective': 0, 'normal': 1},     'Unexpected class mapping. Stop here and inspect the checkpoint.'

model = efficientnet_v2_s(weights=None)
features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(features, 2)
model.load_state_dict(checkpoint['model'], strict=True)
model.eval()

export_path = Path('/kaggle/working/efficientnetv2_s.onnx')
dummy = torch.randn(1, 3, 224, 224, dtype=torch.float32)

with torch.inference_mode():
    torch.onnx.export(
        model, dummy, str(export_path),
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=['images'],
        output_names=['logits'],
        dynamic_axes=None,  # one photograph per inference, 224 x 224
    )

onnx_model = onnx.load(str(export_path))
onnx.checker.check_model(onnx_model)
print('ONNX model saved:', export_path)
print('Size:', round(export_path.stat().st_size / (1024 ** 2), 2), 'MiB')


## 4. Verify ONNX logits against PyTorch logits

In [ ]:
import numpy as np
import onnxruntime as ort

providers = ['CPUExecutionProvider']
ort_session = ort.InferenceSession(str(export_path), providers=providers)
input_name = ort_session.get_inputs()[0].name
output_name = ort_session.get_outputs()[0].name

with torch.inference_mode():
    torch_logits = model(dummy).numpy()
onnx_logits = ort_session.run([output_name], {input_name: dummy.numpy()})[0]
max_diff = np.max(np.abs(torch_logits - onnx_logits))
print('Input:', input_name, ort_session.get_inputs()[0].shape)
print('Output:', output_name, onnx_logits.shape)
print('PyTorch logits:', torch_logits[0])
print('ONNX logits:   ', onnx_logits[0])
print('Maximum absolute difference:', float(max_diff))
if max_diff > 0.01:
    print('WARNING: Larger-than-expected ONNX/PyTorch difference. Verify before deployment.')
else:
    print('Conversion check passed for this input.')


## 5. Download `efficientnetv2_s.onnx`

**Important:** This ~80 MB file must be added to GitHub with **GitHub Desktop**, not the GitHub web uploader (25 MiB limit). Verify the actual file size is below GitHub/Vercel limits. Do not upload the original `.pth` file to the website.

In [ ]:
from IPython.display import FileLink, display
import os
os.chdir('/kaggle/working')
print('Click the link below to download your converted model:')
display(FileLink('efficientnetv2_s.onnx'))
